In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import os
import json

# ============================================================================
# 第一部分：数据预处理（保持Alex的原始逻辑，但添加测试集分割）
# ============================================================================

# 设置随机种子确保可重现性
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 📁 修改：设置保存路径
export_path = './enhanced_alexs_results/'  # 保存在当前目录下
os.makedirs(export_path, exist_ok=True)
os.makedirs(os.path.join(export_path, 'visualizations'), exist_ok=True)

# 📊 数据加载和预处理（增强版）
print("📂 加载数据...")
f = h5py.File('/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat','r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
print(f"原始数据形状: {train_data.shape}")
print(f"原始标签形状: {train_region.shape}")
print(f"prob_idx形状: {prob_idx.shape}")

del arrays, f

# 🔧 修改：创建更科学的数据分割
# 使用prob_idx = 38作为测试集（保持Alex的逻辑）
test_indices = np.where(prob_idx == 38)[0]
train_val_indices = np.where(prob_idx != 38)[0]

test_data = train_data[test_indices, :]
test_labels = train_region[test_indices, :]
train_val_data = train_data[train_val_indices, :]
train_val_labels = train_region[train_val_indices, :]

print(f"测试集形状: {test_data.shape}")
print(f"训练+验证集形状: {train_val_data.shape}")

# 📊 进一步分割训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    train_val_data, train_val_labels, 
    test_size=0.2, random_state=42, stratify=np.argmax(train_val_labels, axis=1)
)

print(f"最终训练集形状: {X_train.shape}")
print(f"最终验证集形状: {X_val.shape}")
print(f"最终测试集形状: {test_data.shape}")

del train_data, train_region, prob_idx, train_val_data, train_val_labels, test_indices, train_val_indices

# 🔧 标准化（只在训练集上拟合）
print("📊 应用标准化...")
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(test_data)

print("✅ 数据预处理完成")


In [ ]:
# ============================================================================
# 第二部分：模型定义（保持Alex的架构，但添加更多配置选项）
# ============================================================================

# 🔧 模型配置（增强版）
CONFIG = {
    'batch_size': 128,
    'epochs': 25,
    'num_classes': 102,
    'input_dim': 341,
    'learning_rate': 0.00001,
    'weight_decay': 0.00001,
    'dropout_rate': 0.5,
    'hidden_dim': 4096,
    'validation_frequency': 1,  # 每个epoch都验证
    'early_stopping_patience': 10,
    'save_best_model': True
}

print(f"🔧 模型配置: {CONFIG}")

# L2正则化函数（保持Alex的逻辑）
def kernel_l2_regularization(model, weight_decay=0.00001):
    """只对权重矩阵应用L2正则化，跳过偏置项"""
    l2_reg = 0
    for name, param in model.named_parameters():
        if 'weight' in name and param.requires_grad:
            l2_reg += torch.norm(param, p=2) ** 2
    return weight_decay * l2_reg

# 🔧 增强版模型类（保持Alex的架构）
class EnhancedRegModel(nn.Module):
    def __init__(self, input_dim=341, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(EnhancedRegModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim) 
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, hidden_dim)
        self.fc5 = nn.Linear(hidden_dim, num_classes)  # visualized_layer
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, x):
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.dropout(F.relu(self.fc3(x)))
        x = self.dropout(F.relu(self.fc4(x)))
        x = self.fc5(x)  # 不应用softmax，让CrossEntropyLoss处理
        return x

model = EnhancedRegModel(
    input_dim=CONFIG['input_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    num_classes=CONFIG['num_classes'],
    dropout_rate=CONFIG['dropout_rate']
).to(device)

print(f"🏗️ 模型创建完成，参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================================
# 第三部分：评估函数（借鉴你的版本）
# ============================================================================

def calculate_gross_accuracy(y_true, y_pred):
    """
    计算Gross Accuracy = 正确分类的体素数量 / 总体素数量 × 100%
    这就是标准的整体准确率
    
    Args:
        y_true: 真实标签 (类别索引，0-101)
        y_pred: 预测标签 (类别索引，0-101)
    
    Returns:
        gross_accuracy: Gross Accuracy (正确分类比例)
        detailed_results: 详细的分类结果
    """
    
    # 计算正确分类的体素数量
    correct_voxels = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    
    # 总体素数量
    total_voxels = len(y_true)
    
    # Gross Accuracy = 正确分类的体素数量 / 总体素数量
    gross_accuracy = correct_voxels / total_voxels if total_voxels > 0 else 0.0
    
    # 计算每个类别的详细统计
    per_class_stats = {}
    unique_classes = sorted(list(set(y_true + y_pred)))
    
    for class_id in unique_classes:
        # 该类别的真实样本索引
        true_indices = [i for i, label in enumerate(y_true) if label == class_id]
        # 该类别被预测为该类别的样本索引
        pred_indices = [i for i, label in enumerate(y_pred) if label == class_id]
        
        # 真正例 (True Positives)
        tp = len([i for i in true_indices if y_pred[i] == class_id])
        # 假正例 (False Positives) 
        fp = len([i for i in pred_indices if y_true[i] != class_id])
        # 假负例 (False Negatives)
        fn = len(true_indices) - tp
        
        # 该类别的准确率 (召回率)
        recall = tp / len(true_indices) if len(true_indices) > 0 else 0.0
        # 精确率
        precision = tp / len(pred_indices) if len(pred_indices) > 0 else 0.0
        # F1分数
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        per_class_stats[class_id] = {
            'true_count': len(true_indices),      # 真实样本数
            'pred_count': len(pred_indices),      # 预测样本数
            'correct_count': tp,                  # 正确预测数
            'recall': recall,                     # 召回率 (该类别的准确率)
            'precision': precision,               # 精确率
            'f1': f1                             # F1分数
        }
    
    detailed_results = {
        'gross_accuracy': gross_accuracy,
        'correct_voxels': correct_voxels,
        'total_voxels': total_voxels,
        'per_class_stats': per_class_stats,
        'unique_classes': unique_classes,
        'error_count': total_voxels - correct_voxels,
        'error_rate': (total_voxels - correct_voxels) / total_voxels if total_voxels > 0 else 0.0
    }
    
    return gross_accuracy, detailed_results

def evaluate_model_comprehensive(model, data_loader, device, dataset_name="Dataset"):
    """
    全面评估模型性能，返回多种指标（包含gross classification accuracy）
    """
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0
    total_samples = 0
    
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for inputs, targets in tqdm(data_loader, desc=f"评估 {dataset_name}", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets) # 假设 criterion 在函数外部或作为参数传入
            
            # 使用 argmax 获取预测类别
            _, predicted = torch.max(outputs.data, 1)

            total_loss += loss.item() * inputs.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    avg_loss = total_loss / len(data_loader.dataset)

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    print(f"\n--- Debugging `accuracy_score` inputs --- ")
    print(f"Type of all_targets: {type(all_targets)}")
    print(f"Shape of all_targets: {all_targets.shape}")
    if all_targets.size > 0:
        print(f"Sample of all_targets (first 5): {all_targets[:5]}")
        print(f"Unique values in all_targets: {np.unique(all_targets)}")
    else:
        print("all_targets is empty.")
    
    print(f"Type of all_preds: {type(all_preds)}")
    print(f"Shape of all_preds: {all_preds.shape}")
    if all_preds.size > 0:
        print(f"Sample of all_preds (first 5): {all_preds[:5]}")
        print(f"Unique values in all_preds: {np.unique(all_preds)}\")")
    else:
        print("all_preds is empty.")
    print(f"--- End Debugging ---\n")
    
    accuracy = accuracy_score(all_targets, all_preds)
    f1_macro = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    f1_weighted = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(all_targets, all_preds)
    balanced_acc = balanced_accuracy_score(all_targets, all_preds)
    
    # 针对多分类F1
    f1_per_class = f1_score(all_targets, all_preds, average=None, zero_division=0)

    # 混淆矩阵
    cm = confusion_matrix(all_targets, all_preds)

    # =========================================================================
    # 🔥 在这里添加 per_label_accuracy 的计算
    # =========================================================================
    per_label_accuracies = {}
    unique_labels = np.unique(all_targets)
    for i, label in enumerate(unique_labels):
        true_positives = cm[i, i]
        total_in_class = np.sum(cm[i, :]) # 真实标签为该类别的总数
        if total_in_class > 0:
            per_label_accuracies[label] = true_positives / total_in_class
        else:
            per_label_accuracies[label] = 0.0 # 如果该类别没有真实样本，则准确率为0

    # 如果你希望 'per_label_accuracy' 作为一个单独的变量，指向这个字典
    per_label_accuracy = per_label_accuracies 
    # =========================================================================

    # Gross Accuracy (体素级别) - 确保已经计算
    # 假设 gross_accuracy 和 gross_details 已经在前面的代码中计算
    # 如果没有，你需要在评估循环中或之后补充这部分的计算逻辑。
    # 这里只是占位符，如果你的代码中有，它会按预期工作。
    gross_accuracy = accuracy # 举例，你需要根据你的定义进行计算evaluate_model_comprehensive
    gross_details = {} # 举例，你需要根据你的定义进行填充


    results = {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'kappa': kappa,
        'balanced_accuracy': balanced_acc,
        'loss': avg_loss,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'targets': all_targets,
        'unique_classes': np.unique(all_targets),
        'gross_accuracy': gross_accuracy,
        'gross_details': gross_details,
        'per_label_accuracy': per_label_accuracy,         # ⭐️ 新加的
        'per_label_accuracies': per_label_accuracies,     # (可选) 各类别准确率
    }

    return results


In [ ]:
# ============================================================================
# 第四部分：可视化函数（借鉴你的丰富版本）
# ============================================================================

def plot_training_history_comprehensive(history, save_path):
    """
    绘制综合训练历史，包含所有指标（包括gross classification accuracy）
    """
    fig, axes = plt.subplots(3, 3, figsize=(20, 18))  # 🔥 修改为3x3布局
    
    epochs = range(1, len(history['train_loss']) + 1)
    val_epochs = range(1, len(history['val_loss']) + 1)
    
    # 1. 损失曲线
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Training Loss')
    axes[0, 0].plot(val_epochs, history['val_loss'], 'g-', linewidth=2, marker='o', markersize=4, label='Validation Loss')
    if 'test_loss' in history and len(history['test_loss']) > 0:
        axes[0, 0].plot(val_epochs, history['test_loss'], 'r--', linewidth=2, marker='s', markersize=4, label='Test Loss (Monitor)')
    axes[0, 0].set_title('Loss Curves', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 细分类准确率曲线 (Fine-grained Accuracy)
    axes[0, 1].plot(epochs, history['train_accuracy'], 'b-', linewidth=2, label='Training Accuracy')
    axes[0, 1].plot(val_epochs, history['val_accuracy'], 'g-', linewidth=2, marker='o', markersize=4, label='Validation Accuracy')
    if 'test_accuracy' in history and len(history['test_accuracy']) > 0:
        axes[0, 1].plot(val_epochs, history['test_accuracy'], 'r--', linewidth=2, marker='s', markersize=4, label='Test Accuracy (Monitor)')
    axes[0, 1].set_title('Fine-grained Accuracy (102 Classes)', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 🔥 3. Gross Accuracy曲线 = 正确分类的体素数量 / 总体素数量
    axes[0, 2].plot(val_epochs, history['val_gross_accuracy'], 'g-', linewidth=3, marker='o', markersize=6, 
                    markerfacecolor='lightgreen', markeredgewidth=2, label='Validation Gross Accuracy')
    if 'test_gross_accuracy' in history and len(history['test_gross_accuracy']) > 0:
        axes[0, 2].plot(val_epochs, history['test_gross_accuracy'], 'r--', linewidth=3, marker='s', markersize=6, 
                        markerfacecolor='lightcoral', markeredgewidth=2, label='Test Gross Accuracy (Monitor)')
    
    # 标注最佳点
    if history['val_gross_accuracy']:
        best_idx = np.argmax(history['val_gross_accuracy'])
        best_epoch = val_epochs[best_idx]
        best_val_gross = history['val_gross_accuracy'][best_idx]
        axes[0, 2].scatter([best_epoch], [best_val_gross], color='darkgreen', s=150, marker='*', zorder=5)
        
        if 'test_gross_accuracy' in history and len(history['test_gross_accuracy']) > best_idx:
            best_test_gross = history['test_gross_accuracy'][best_idx]
            axes[0, 2].scatter([best_epoch], [best_test_gross], color='darkred', s=150, marker='*', zorder=5)
    
    axes[0, 2].set_title('Gross Accuracy (正确体素数/总体素数)', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Gross Accuracy')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. F1分数曲线
    axes[1, 0].plot(val_epochs, history['val_f1_macro'], 'g-', linewidth=2, marker='o', markersize=4, label='Validation F1 Macro')
    if 'test_f1_macro' in history and len(history['test_f1_macro']) > 0:
        axes[1, 0].plot(val_epochs, history['test_f1_macro'], 'r--', linewidth=2, marker='s', markersize=4, label='Test F1 Macro (Monitor)')
    
    # 标注最佳点
    if history['val_f1_macro']:
        best_idx = np.argmax(history['val_f1_macro'])
        best_epoch = val_epochs[best_idx]
        best_val_f1 = history['val_f1_macro'][best_idx]
        axes[1, 0].scatter([best_epoch], [best_val_f1], color='green', s=100, marker='*', zorder=5)
        
        if 'test_f1_macro' in history and len(history['test_f1_macro']) > best_idx:
            best_test_f1 = history['test_f1_macro'][best_idx]
            axes[1, 0].scatter([best_epoch], [best_test_f1], color='red', s=100, marker='*', zorder=5)
    
    axes[1, 0].set_title('F1 Macro Score', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1 Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. 其他验证指标
    if 'val_kappa' in history:
        axes[1, 1].plot(val_epochs, history['val_kappa'], 'purple', linewidth=2, marker='d', markersize=4, label="Cohen's Kappa")
    if 'val_balanced_accuracy' in history:
        axes[1, 1].plot(val_epochs, history['val_balanced_accuracy'], 'orange', linewidth=2, marker='^', markersize=4, label='Balanced Accuracy')
    axes[1, 1].set_title('Additional Metrics', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 🔥 6. 显示Gross Accuracy实际上等于Overall Accuracy的验证
    axes[1, 2].plot(val_epochs, history['val_accuracy'], 'b-', linewidth=2, marker='o', markersize=4, label='Overall Accuracy (sklearn)')
    axes[1, 2].plot(val_epochs, history['val_gross_accuracy'], 'g--', linewidth=2, marker='s', markersize=4, label='Gross Accuracy (手动计算)')
    
    # 计算差异（应该为0或接近0）
    if history['val_accuracy'] and history['val_gross_accuracy']:
        differences = [(gross - overall) for overall, gross in zip(history['val_accuracy'], history['val_gross_accuracy'])]
        max_diff = max(abs(d) for d in differences) if differences else 0
        axes[1, 2].text(0.05, 0.95, f'Max Difference: {max_diff:.6f}', 
                       transform=axes[1, 2].transAxes, fontsize=10, 
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    axes[1, 2].set_title('验证: Gross Accuracy = Overall Accuracy', fontsize=14, fontweight='bold')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Accuracy')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    # 7. 性能差异分析 - Fine Accuracy
    if 'val_accuracy' in history and 'test_accuracy' in history and len(history['test_accuracy']) > 0:
        acc_gaps = [v - t for v, t in zip(history['val_accuracy'], history['test_accuracy']) if v is not None and t is not None]
        if acc_gaps:
            axes[2, 0].plot(range(1, len(acc_gaps) + 1), acc_gaps, 'blue', linewidth=2, marker='d', markersize=4)
            axes[2, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)
            axes[2, 0].axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='Potential Overfitting')
            axes[2, 0].set_title('Fine Accuracy Gap (Val - Test)', fontsize=14, fontweight='bold')
            axes[2, 0].set_xlabel('Epoch')
            axes[2, 0].set_ylabel('Accuracy Gap')
            axes[2, 0].legend()
            axes[2, 0].grid(True, alpha=0.3)
    
    # 🔥 8. Gross Accuracy差异分析  
    if 'val_gross_accuracy' in history and 'test_gross_accuracy' in history and len(history['test_gross_accuracy']) > 0:
        gross_gaps = [v - t for v, t in zip(history['val_gross_accuracy'], history['test_gross_accuracy']) if v is not None and t is not None]
        if gross_gaps:
            axes[2, 1].plot(range(1, len(gross_gaps) + 1), gross_gaps, 'green', linewidth=2, marker='d', markersize=4)
            axes[2, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
            axes[2, 1].axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='Potential Overfitting')
            axes[2, 1].set_title('Gross Accuracy Gap (Val - Test)', fontsize=14, fontweight='bold')
            axes[2, 1].set_xlabel('Epoch')
            axes[2, 1].set_ylabel('Gross Accuracy Gap')
            axes[2, 1].legend()
            axes[2, 1].grid(True, alpha=0.3)
    
    # 9. 学习率或损失差异分析
    if 'learning_rate' in history:
        axes[2, 2].plot(epochs, history['learning_rate'], 'red', linewidth=2)
        axes[2, 2].set_title('Learning Rate', fontsize=14, fontweight='bold')
        axes[2, 2].set_xlabel('Epoch')
        axes[2, 2].set_ylabel('Learning Rate')
        axes[2, 2].set_yscale('log')
        axes[2, 2].grid(True, alpha=0.3)
    else:
        # 显示损失比较
        if 'train_loss' in history and 'val_loss' in history:
            loss_gaps = [t - v for t, v in zip(history['train_loss'][:len(history['val_loss'])], history['val_loss'])]
            axes[2, 2].plot(range(1, len(loss_gaps) + 1), loss_gaps, 'blue', linewidth=2, marker='o', markersize=4)
            axes[2, 2].axhline(y=0, color='black', linestyle='-', alpha=0.5)
            axes[2, 2].set_title('Train-Val Loss Gap', fontsize=14, fontweight='bold')
            axes[2, 2].set_xlabel('Epoch')
            axes[2, 2].set_ylabel('Loss Gap')
            axes[2, 2].grid(True, alpha=0.3)
    
    plt.tight_layout(pad=3.0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # 🔥 打印关键统计信息（包含gross accuracy）
    if history['val_f1_macro']:
        best_val_f1 = max(history['val_f1_macro'])
        best_idx = history['val_f1_macro'].index(best_val_f1)
        print(f"\n📊 训练总结:")
        print(f"  最佳验证F1: {best_val_f1:.4f} (Epoch {best_idx + 1})")
        
        if 'test_f1_macro' in history and len(history['test_f1_macro']) > best_idx:
            corresponding_test_f1 = history['test_f1_macro'][best_idx]
            print(f"  对应测试F1: {corresponding_test_f1:.4f}")
            print(f"  F1性能差距: {best_val_f1 - corresponding_test_f1:+.4f}")
        
        # 🔥 修改per-label accuracy总结
        if 'val_per_label_accuracy' in history and len(history['val_per_label_accuracy']) > best_idx:
            corresponding_val_per_label = history['val_per_label_accuracy'][best_idx]
            print(f"  对应验证Per-Label准确率: {corresponding_val_per_label:.4f}")
            
            if 'test_per_label_accuracy' in history and len(history['test_per_label_accuracy']) > best_idx:
                corresponding_test_per_label = history['test_per_label_accuracy'][best_idx]
                print(f"  对应测试Per-Label准确率: {corresponding_test_per_label:.4f}")
                print(f"  Per-Label准确率差距: {corresponding_val_per_label - corresponding_test_per_label:+.4f}")
        
        # 最佳per-label accuracy的epoch
        if 'val_per_label_accuracy' in history:
            best_per_label_idx = np.argmax(history['val_per_label_accuracy'])
            best_per_label_accuracy = history['val_per_label_accuracy'][best_per_label_idx]
            print(f"  最佳验证Per-Label准确率: {best_per_label_accuracy:.4f} (Epoch {best_per_label_idx + 1})")
        axes[1, 0].hist(valid_accuracies, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
        axes[1, 0].axvline(np.mean(valid_accuracies), color='red', linestyle='--', 
                          label=f'Mean: {np.mean(valid_accuracies):.3f}')
        axes[1, 0].axvline(np.median(valid_accuracies), color='orange', linestyle='--', 
                          label=f'Median: {np.median(valid_accuracies):.3f}')
        axes[1, 0].set_title('Distribution of Per-Label Accuracies', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Accuracy')
        axes[1, 0].set_ylabel('Number of Labels')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # 5. 最佳和最差表现的类别（基于验证集）
    if 'per_label_details' in val_results:
        val_per_label_details = val_results['per_label_details']
        label_accuracies = val_per_label_details['per_label_accuracies']
        label_support = val_per_label_details['per_label_support']
        
        # 找到有足够样本的类别（至少5个样本）
        sufficient_labels = [(i, acc) for i, (acc, sup) in enumerate(zip(label_accuracies, label_support)) if sup >= 5]
        
        if len(sufficient_labels) >= 10:
            # 排序并取最好和最差的各5个
            sufficient_labels.sort(key=lambda x: x[1])
            worst_labels = sufficient_labels[:5]
            best_labels = sufficient_labels[-5:]
            
            # 绘制条形图
            x_pos = np.arange(5)
            worst_accs = [acc for _, acc in worst_labels]
            best_accs = [acc for _, acc in best_labels]
            worst_indices = [idx for idx, _ in worst_labels]
            best_indices = [idx for idx, _ in best_labels]
            
            bars1 = axes[1, 1].bar(x_pos - 0.2, worst_accs, 0.4, label='Worst 5', color='red', alpha=0.7)
            bars2 = axes[1, 1].bar(x_pos + 0.2, best_accs, 0.4, label='Best 5', color='green', alpha=0.7)
            
            # 添加标签索引
            for i, (bar, idx) in enumerate(zip(bars1, worst_indices)):
                axes[1, 1].text(bar.get_x() + bar.get_width()/2., -0.05, 
                               f'L{idx}', ha='center', va='top', rotation=45, fontsize=8)
            
            for i, (bar, idx) in enumerate(zip(bars2, best_indices)):
                axes[1, 1].text(bar.get_x() + bar.get_width()/2., -0.05, 
                               f'L{idx}', ha='center', va='top', rotation=45, fontsize=8)
            
            axes[1, 1].set_title('Best vs Worst Performing Labels', fontsize=14, fontweight='bold')
            axes[1, 1].set_ylabel('Accuracy')
            axes[1, 1].set_xticks(x_pos)
            axes[1, 1].set_xticklabels([f'#{i+1}' for i in range(5)])
            axes[1, 1].legend()
            axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    # 6. 统计信息摘要
    axes[1, 2].axis('off')  # 关闭坐标轴，用于显示文本统计
    
    # 计算统计信息
    stats_text = "📊 Per-Label Performance Statistics\n\n"
    
    for name in dataset_names:
        overall_acc = overall_accuracies[name]
        per_label_acc = per_label_accuracies[name]
        difference = per_label_acc - overall_acc
        
        stats_text += f"{name} Dataset:\n"
        stats_text += f"  Overall Accuracy: {overall_acc:.4f}\n"
        stats_text += f"  Per-Label Accuracy: {per_label_acc:.4f}\n"
        stats_text += f"  Difference: {difference:+.4f}\n\n"
    
    # 基于验证集的详细统计
    if 'per_label_details' in val_results:
        val_details = val_results['per_label_details']
        
        # 有数据的类别统计
        labels_with_data = val_details['labels_with_data']
        labels_without_data = val_details['labels_without_data']
        
        stats_text += f"Label Coverage (Validation):\n"
        stats_text += f"  Labels with data: {len(labels_with_data)}/{CONFIG['num_classes']}\n"
        stats_text += f"  Labels without data: {len(labels_without_data)}\n\n"
        
        # 准确率统计
        valid_accuracies = [val_details['per_label_accuracies'][i] for i in labels_with_data]
        if valid_accuracies:
            stats_text += f"Accuracy Statistics (Labels with data):\n"
            stats_text += f"  Mean: {np.mean(valid_accuracies):.4f}\n"
            stats_text += f"  Std: {np.std(valid_accuracies):.4f}\n"
            stats_text += f"  Min: {np.min(valid_accuracies):.4f}\n"
            stats_text += f"  Max: {np.max(valid_accuracies):.4f}\n"
            stats_text += f"  Median: {np.median(valid_accuracies):.4f}\n\n"
        
        # 支持度统计
        valid_support = [val_details['per_label_support'][i] for i in labels_with_data]
        if valid_support:
            stats_text += f"Support Statistics:\n"
            stats_text += f"  Total samples: {sum(valid_support)}\n"
            stats_text += f"  Mean per label: {np.mean(valid_support):.1f}\n"
            stats_text += f"  Min support: {np.min(valid_support)}\n"
            stats_text += f"  Max support: {np.max(valid_support)}\n"
        
        # 性能极值
        if len(labels_with_data) > 0:
            worst_label_idx = labels_with_data[np.argmin([val_details['per_label_accuracies'][i] for i in labels_with_data])]
            best_label_idx = labels_with_data[np.argmax([val_details['per_label_accuracies'][i] for i in labels_with_data])]
            
            stats_text += f"\nPerformance Extremes:\n"
            stats_text += f"  Best: Label {best_label_idx} ({val_details['per_label_accuracies'][best_label_idx]:.4f})\n"
            stats_text += f"  Worst: Label {worst_label_idx} ({val_details['per_label_accuracies'][worst_label_idx]:.4f})\n"
    
    axes[1, 2].text(0.05, 0.95, stats_text, transform=axes[1, 2].transAxes,
                   fontsize=10, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return stats_text
    """
    🔥 新增：专门的Gross Classification Analysis可视化
    """
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    datasets = {"Train": train_results, "Validation": val_results, "Test": test_results}
    colors = {'Train': 'blue', 'Validation': 'green', 'Test': 'red'}
    
    # 提取gross classification数据
    gross_accuracies = {}
    gross_class_accuracies = {}
    tissue_groups = None
    
    for name, results in datasets.items():
        if 'gross_details' in results:
            gross_accuracies[name] = results['gross_accuracy']
            gross_class_accuracies[name] = results['gross_details']['gross_class_accuracies']
            if tissue_groups is None:
                tissue_groups = results['gross_details']['gross_labels']
    
    # 1. 整体Gross Accuracy对比
    dataset_names = list(gross_accuracies.keys())
    overall_scores = list(gross_accuracies.values())
    
    bars = axes[0, 0].bar(dataset_names, overall_scores, 
                         color=[colors[name] for name in dataset_names], alpha=0.7)
    axes[0, 0].set_title('Overall Gross Classification Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('Gross Accuracy')
    axes[0, 0].set_ylim(0, 1.0)
    
    # 添加数值标签
    for bar, score in zip(bars, overall_scores):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
    
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # 2. 每个组织群的准确率对比
    if tissue_groups and gross_class_accuracies:
        x_pos = np.arange(len(tissue_groups))
        width = 0.25
        
        for i, (name, class_accs) in enumerate(gross_class_accuracies.items()):
            tissue_scores = [class_accs.get(tissue, 0) for tissue in tissue_groups]
            axes[0, 1].bar(x_pos + i * width, tissue_scores, width, 
                          label=name, color=colors[name], alpha=0.7)
        
        axes[0, 1].set_title('Accuracy by Tissue Group', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Tissue Groups')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].set_xticks(x_pos + width)
        axes[0, 1].set_xticklabels(tissue_groups, rotation=45, ha='right')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # 3. Fine vs Gross Accuracy提升分析
    fine_accuracies = {name: results['accuracy'] for name, results in datasets.items()}
    
    dataset_names = list(fine_accuracies.keys())
    fine_scores = [fine_accuracies[name] for name in dataset_names]
    gross_scores = [gross_accuracies[name] for name in dataset_names]
    improvements = [gross - fine for gross, fine in zip(gross_scores, fine_scores)]
    
    x_pos = np.arange(len(dataset_names))
    width = 0.35
    
    bars1 = axes[0, 2].bar(x_pos - width/2, fine_scores, width, 
                          label='Fine-grained (102 classes)', alpha=0.7, color='lightblue')
    bars2 = axes[0, 2].bar(x_pos + width/2, gross_scores, width, 
                          label='Gross (6 tissue groups)', alpha=0.7, color='lightgreen')
    
    # 添加提升幅度标注
    for i, improvement in enumerate(improvements):
        axes[0, 2].annotate(f'+{improvement:.3f}', 
                           xy=(i, gross_scores[i]), xytext=(i, gross_scores[i] + 0.05),
                           ha='center', va='bottom', fontweight='bold', color='red',
                           arrowprops=dict(arrowstyle='->', color='red', lw=1))
    
    axes[0, 2].set_title('Fine vs Gross Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('Dataset')
    axes[0, 2].set_ylabel('Accuracy')
    axes[0, 2].set_xticks(x_pos)
    axes[0, 2].set_xticklabels(dataset_names)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3, axis='y')
    
    # 4. 组织群混淆矩阵（使用验证集）
    if 'gross_details' in val_results:
        val_gross_details = val_results['gross_details']
        true_gross = val_gross_details['true_gross_labels']
        pred_gross = val_gross_details['pred_gross_labels']
        
        # 创建gross confusion matrix
        from sklearn.metrics import confusion_matrix
        gross_cm = confusion_matrix(true_gross, pred_gross, labels=tissue_groups)
        gross_cm_norm = gross_cm.astype('float') / gross_cm.sum(axis=1)[:, np.newaxis]
        
        im = axes[1, 0].imshow(gross_cm_norm, interpolation='nearest', cmap='Blues')
        axes[1, 0].set_title('Gross Classification Confusion Matrix\n(Validation Set)', fontsize=12, fontweight='bold')
        
        # 添加数值标签
        thresh = gross_cm_norm.max() / 2.
        for i in range(gross_cm_norm.shape[0]):
            for j in range(gross_cm_norm.shape[1]):
                axes[1, 0].text(j, i, f'{gross_cm_norm[i, j]:.2f}',
                               ha="center", va="center",
                               color="white" if gross_cm_norm[i, j] > thresh else "black")
        
        axes[1, 0].set_xticks(range(len(tissue_groups)))
        axes[1, 0].set_yticks(range(len(tissue_groups)))
        axes[1, 0].set_xticklabels(tissue_groups, rotation=45, ha='right')
        axes[1, 0].set_yticklabels(tissue_groups)
        axes[1, 0].set_xlabel('Predicted')
        axes[1, 0].set_ylabel('True')
        
        # 添加colorbar
        plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    # 5. 样本分布分析
    if tissue_groups and 'gross_details' in val_results:
        val_gross_details = val_results['gross_details']
        true_counts = val_gross_details['true_counts']
        pred_counts = val_gross_details['pred_counts']
        
        tissue_true_counts = [true_counts.get(tissue, 0) for tissue in tissue_groups]
        tissue_pred_counts = [pred_counts.get(tissue, 0) for tissue in tissue_groups]
        
        x_pos = np.arange(len(tissue_groups))
        width = 0.35
        
        axes[1, 1].bar(x_pos - width/2, tissue_true_counts, width, 
                      label='True Distribution', alpha=0.7, color='skyblue')
        axes[1, 1].bar(x_pos + width/2, tissue_pred_counts, width, 
                      label='Predicted Distribution', alpha=0.7, color='lightcoral')
        
        axes[1, 1].set_title('Sample Distribution by Tissue Group\n(Validation Set)', fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel('Tissue Groups')
        axes[1, 1].set_ylabel('Number of Samples')
        axes[1, 1].set_xticks(x_pos)
        axes[1, 1].set_xticklabels(tissue_groups, rotation=45, ha='right')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    # 6. 性能提升统计
    axes[1, 2].axis('off')  # 关闭坐标轴，用于显示文本统计
    
    # 计算统计信息
    stats_text = "📊 Gross Classification Statistics\n\n"
    
    for name in dataset_names:
        fine_acc = fine_accuracies[name]
        gross_acc = gross_accuracies[name]
        improvement = gross_acc - fine_acc
        improvement_pct = (improvement / fine_acc) * 100
        
        stats_text += f"{name} Dataset:\n"
        stats_text += f"  Fine Accuracy: {fine_acc:.4f}\n"
        stats_text += f"  Gross Accuracy: {gross_acc:.4f}\n"
        stats_text += f"  Improvement: +{improvement:.4f} ({improvement_pct:+.1f}%)\n\n"
    
    # 整体统计
    avg_fine = np.mean(list(fine_accuracies.values()))
    avg_gross = np.mean(list(gross_accuracies.values()))
    avg_improvement = avg_gross - avg_fine
    
    stats_text += f"Average Across Datasets:\n"
    stats_text += f"  Fine Accuracy: {avg_fine:.4f}\n"
    stats_text += f"  Gross Accuracy: {avg_gross:.4f}\n"
    stats_text += f"  Average Improvement: +{avg_improvement:.4f}\n\n"
    
    # 组织群性能（基于验证集）
    if 'gross_details' in val_results:
        val_gross_class_accs = val_results['gross_details']['gross_class_accuracies']
        best_tissue = max(val_gross_class_accs, key=val_gross_class_accs.get)
        worst_tissue = min(val_gross_class_accs, key=val_gross_class_accs.get)
        
        stats_text += f"Tissue Group Performance (Validation):\n"
        stats_text += f"  Best: {best_tissue} ({val_gross_class_accs[best_tissue]:.4f})\n"
        stats_text += f"  Worst: {worst_tissue} ({val_gross_class_accs[worst_tissue]:.4f})\n"
    
    axes[1, 2].text(0.05, 0.95, stats_text, transform=axes[1, 2].transAxes,
                   fontsize=11, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return stats_text

def plot_confusion_matrix(cm, save_path, title="Confusion Matrix"):
    """绘制混淆矩阵"""
    plt.figure(figsize=(12, 10))
    
    # 使用对数缩放以更好地显示
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # 绘制混淆矩阵
    sns.heatmap(cm_normalized, annot=False, cmap='Blues', cbar=True)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Labels', fontsize=12)
    plt.ylabel('True Labels', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def analyze_class_performance(results_dict, save_path):
    """分析每个类别的性能"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    datasets = list(results_dict.keys())
    colors = ['blue', 'green', 'red']
    
    # 1. F1分数分布
    for i, (dataset, results) in enumerate(results_dict.items()):
        f1_scores = results['f1_per_class']
        axes[0, 0].hist(f1_scores, bins=20, alpha=0.7, color=colors[i], 
                       label=f'{dataset} (mean: {np.mean(f1_scores):.3f})')
    
    axes[0, 0].set_title('F1 Score Distribution by Class', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('F1 Score')
    axes[0, 0].set_ylabel('Number of Classes')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 排序的F1分数
    for i, (dataset, results) in enumerate(results_dict.items()):
        f1_sorted = np.sort(results['f1_per_class'])
        axes[0, 1].plot(f1_sorted, colors[i], linewidth=2, marker='o', markersize=3, label=dataset)
    
    axes[0, 1].set_title('F1 Scores Sorted by Performance', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Class Rank (sorted)')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 数据集间F1相关性（如果有多个数据集）
    if len(datasets) >= 2:
        f1_train = results_dict[datasets[0]]['f1_per_class']
        f1_val = results_dict[datasets[1]]['f1_per_class']
        
        # 确保长度一致
        min_len = min(len(f1_train), len(f1_val))
        f1_train = f1_train[:min_len]
        f1_val = f1_val[:min_len]
        
        axes[1, 0].scatter(f1_train, f1_val, alpha=0.6, s=30)
        axes[1, 0].plot([0, 1], [0, 1], 'r--', alpha=0.8)
        
        correlation = np.corrcoef(f1_train, f1_val)[0, 1]
        axes[1, 0].set_title(f'{datasets[0]} vs {datasets[1]} F1 Correlation: {correlation:.3f}', 
                            fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel(f'{datasets[0]} F1 Score')
        axes[1, 0].set_ylabel(f'{datasets[1]} F1 Score')
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. 最差和最好的类别
    val_results = results_dict.get('Validation', results_dict[list(results_dict.keys())[0]])
    f1_scores = val_results['f1_per_class']
    
    worst_classes = np.argsort(f1_scores)[:10]
    best_classes = np.argsort(f1_scores)[-10:]
    
    x_pos = np.arange(10)
    axes[1, 1].bar(x_pos - 0.2, f1_scores[worst_classes], 0.4, label='Worst 10', color='red', alpha=0.7)
    axes[1, 1].bar(x_pos + 0.2, f1_scores[best_classes], 0.4, label='Best 10', color='green', alpha=0.7)
    
    axes[1, 1].set_title('Best vs Worst Performing Classes', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Class Rank')
    axes[1, 1].set_ylabel('F1 Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# ============================================================================
# 第五部分：增强版训练循环（集成监控功能）
# ============================================================================

def enhanced_training_loop():
    """
    增强版训练循环，包含全面的监控和评估
    """
    # 准备数据加载器
    train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_val))
    test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled), torch.FloatTensor(test_labels))
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    # 优化器和损失函数
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    criterion = nn.CrossEntropyLoss()
    
    # 训练历史记录
    history = {
        'train_loss': [], 'train_accuracy': [],
        'val_loss': [], 'val_accuracy': [], 'val_f1_macro': [], 'val_f1_weighted': [],
        'val_kappa': [], 'val_balanced_accuracy': [], 'val_per_label_accuracy': [],
        'val_gross_accuracy': [],  # 添加这行
        'test_loss': [], 'test_accuracy': [], 'test_f1_macro': [], 'test_f1_weighted': [],
        'test_per_label_accuracy': [], 'test_gross_accuracy': [],  # 添加 test_gross_accuracy
        'learning_rate': []
    }
    
    best_val_f1 = 0.0
    best_model_state = None
    patience_counter = 0
    
    print(f"🚀 开始训练 - 总共 {CONFIG['epochs']} 个epoch")
    train_start_time = time.time()
    
    for epoch in range(CONFIG['epochs']):
        epoch_start_time = time.time()
        
        # ==================== 训练阶段 ====================
        model.train()
        epoch_train_loss = 0
        train_correct = 0
        train_total = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Train]", leave=False)
        
        for batch_idx, (data, target) in enumerate(train_pbar):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            
            # 将one-hot编码转换为类别索引
            target_indices = torch.argmax(target, dim=1)
            
            # 计算损失（包含L2正则化）
            base_loss = criterion(output, target_indices)
            l2_reg = kernel_l2_regularization(model, weight_decay=CONFIG['weight_decay'])
            total_loss = base_loss + l2_reg
            
            total_loss.backward()
            optimizer.step()
            
            # 统计
            epoch_train_loss += total_loss.item()
            predicted = torch.argmax(output, dim=1)
            train_total += target_indices.size(0)
            train_correct += (predicted == target_indices).sum().item()
            
            # 更新进度条
            if batch_idx % 50 == 0:
                train_pbar.set_postfix({
                    'Loss': f'{total_loss.item():.4f}',
                    'Acc': f'{100. * train_correct / train_total:.2f}%'
                })
        
        # 记录训练指标
        avg_train_loss = epoch_train_loss / len(train_loader)
        train_accuracy = train_correct / train_total
        
        history['train_loss'].append(avg_train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['learning_rate'].append(optimizer.param_groups[0]['lr'])
        
        # ==================== 验证阶段 ====================
        if (epoch + 1) % CONFIG['validation_frequency'] == 0:
            print(f"\n📊 Epoch {epoch+1} 验证...")
            
            # 验证集评估
            val_results = evaluate_model_comprehensive(model, val_loader, device, "Validation")
            
            # 测试集监控（不参与模型选择）
            test_results = evaluate_model_comprehensive(model, test_loader, device, "Test")
        
            # 记录验证指标
            history['val_loss'].append(val_results['loss'])
            history['val_accuracy'].append(val_results['accuracy'])
            history['val_f1_macro'].append(val_results['f1_macro'])
            history['val_f1_weighted'].append(val_results['f1_weighted'])
            history['val_kappa'].append(val_results['kappa'])
            history['val_balanced_accuracy'].append(val_results['balanced_accuracy'])
            history['val_per_label_accuracy'].append(val_results['per_label_accuracy'])
            history['val_gross_accuracy'].append(val_results['gross_accuracy'])  # 添加这行

            # 记录测试指标
            history['test_loss'].append(test_results['loss'])
            history['test_accuracy'].append(test_results['accuracy'])
            history['test_f1_macro'].append(test_results['f1_macro'])
            history['test_f1_weighted'].append(test_results['f1_weighted'])
            history['test_per_label_accuracy'].append(test_results['per_label_accuracy'])
            history['test_gross_accuracy'].append(test_results['gross_accuracy'])  # 添加这行

            
            # 🏆 模型保存逻辑
            if val_results['f1_macro'] > best_val_f1:
                best_val_f1 = val_results['f1_macro']
                best_model_state = model.state_dict().copy()
                patience_counter = 0
                
                # 保存最佳模型
                if CONFIG['save_best_model']:
                    torch.save({
                        'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'best_val_f1': best_val_f1,
                        'config': CONFIG,
                        'scaler_mean': scaler.mean_,
                        'scaler_scale': scaler.scale_,
                    }, os.path.join(export_path, 'best_model_enhanced.pth'))
                
                print(f"   ✅ 新的最佳模型! Val F1: {best_val_f1:.4f}, Test F1: {test_results['f1_macro']:.4f}")
            else:
                patience_counter += 1
            
            # 打印详细结果（包含per-label accuracy）
            epoch_time = time.time() - epoch_start_time
            print(f"Epoch {epoch+1}/{CONFIG['epochs']} - 用时: {epoch_time:.2f}s")
            print(f"  训练 - Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.4f}")
            print(f"  验证 - Loss: {val_results['loss']:.4f}, Acc: {val_results['accuracy']:.4f}, F1: {val_results['f1_macro']:.4f}, Kappa: {val_results['kappa']:.4f}")
            print(f"         Per-Label Acc: {val_results['per_label_accuracy']:.4f}")  # 🔥 修改
            print(f"  测试 - Loss: {test_results['loss']:.4f}, Acc: {test_results['accuracy']:.4f}, F1: {test_results['f1_macro']:.4f} (监控)")
            print(f"         Per-Label Acc: {test_results['per_label_accuracy']:.4f} (监控)")  # 🔥 修改
            print(f"  F1差异 - Val-Test: {val_results['f1_macro'] - test_results['f1_macro']:+.4f}")
            print(f"  Per-Label差异 - Val-Test: {val_results['per_label_accuracy'] - test_results['per_label_accuracy']:+.4f}")  # 🔥 修改
            
            # 早停检查
            if patience_counter >= CONFIG['early_stopping_patience']:
                print(f"🛑 早停触发! 验证F1已连续{CONFIG['early_stopping_patience']}个epoch未改善")
                break
        
        else:
            # 只记录训练指标
            print(f"Epoch {epoch+1}/{CONFIG['epochs']} - Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.4f}")
    
    total_time = time.time() - train_start_time
    print(f"\n🎉 训练完成! 总用时: {total_time:.2f}秒")
    print(f"🏆 最佳验证F1: {best_val_f1:.4f}")
    
    # 恢复最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("✅ 已恢复最佳模型权重")
    
    return history


In [ ]:

# ============================================================================
# 第六部分：最终评估和完整可视化
# ============================================================================

def final_comprehensive_evaluation():
    """
    最终的全面评估，生成所有可视化和报告
    """
    print("\n📊 开始最终全面评估...")
    
    # 准备数据加载器
    train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_val))
    test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled), torch.FloatTensor(test_labels))
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    # 评估所有数据集
    print("🔍 评估训练集...")
    train_results = evaluate_model_comprehensive(model, train_loader, device, "Train")
    
    print("🔍 评估验证集...")
    val_results = evaluate_model_comprehensive(model, val_loader, device, "Validation")
    
    print("🔍 评估测试集...")
    test_results = evaluate_model_comprehensive(model, test_loader, device, "Test")
    
    # 打印最终结果对比
    print(f"\n📋 最终性能对比:")
    print(f"{'数据集':<12} {'准确率':<8} {'F1宏平均':<10} {'F1加权':<10} {'平衡准确率':<12} {'Kappa':<8} {'损失':<8}")
    print("-" * 80)
    
    datasets = [("训练集", train_results), ("验证集", val_results), ("测试集", test_results)]
    for name, results in datasets:
        print(f"{name:<12} {results['accuracy']:<8.4f} {results['f1_macro']:<10.4f} "
              f"{results['f1_weighted']:<10.4f} {results['balanced_accuracy']:<12.4f} "
              f"{results['kappa']:<8.4f} {results['loss']:<8.4f}")
    
    # 生成可视化
    results_dict = {"Train": train_results, "Validation": val_results, "Test": test_results}
    
    # 1. 混淆矩阵
    print("\n📈 生成混淆矩阵...")
    for name, results in datasets:
        cm_path = os.path.join(export_path, 'visualizations', f'confusion_matrix_{name.lower()}.png')
        plot_confusion_matrix(results['confusion_matrix'], cm_path, f'{name} Confusion Matrix')
    
    # 2. 类别性能分析
    print("📈 生成类别性能分析...")
    class_analysis_path = os.path.join(export_path, 'visualizations', 'class_performance_analysis.png')
    analyze_class_performance(results_dict, class_analysis_path)
    
    # 🔥 3. 修改：Per-Label Performance Analysis
    print("📈 生成Per-Label Performance分析...")
    per_label_analysis_path = os.path.join(export_path, 'visualizations', 'per_label_performance_analysis.png')
    per_label_stats = plot_per_label_performance_analysis(train_results, val_results, test_results, per_label_analysis_path)
    
    # 3. 生成详细报告
    print("📝 生成详细报告...")
    generate_detailed_report(train_results, val_results, test_results)
    
    return train_results, val_results, test_results

def generate_detailed_report(train_results, val_results, test_results):
    """生成详细的文本报告（包含gross classification analysis）"""
    
    report_path = os.path.join(export_path, 'detailed_evaluation_report.txt')
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("Alex版本增强训练 - 详细评估报告\n")
        f.write("=" * 60 + "\n\n")
        
        f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"PyTorch版本: {torch.__version__}\n")
        f.write(f"使用设备: {device}\n\n")
        
        # 模型配置
        f.write("模型配置:\n")
        f.write("-" * 30 + "\n")
        for key, value in CONFIG.items():
            f.write(f"{key}: {value}\n")
        f.write("\n")
        
        # 数据集信息
        f.write("数据集信息:\n")
        f.write("-" * 30 + "\n")
        f.write(f"训练集样本数: {len(X_train_scaled)}\n")
        f.write(f"验证集样本数: {len(X_val_scaled)}\n")
        f.write(f"测试集样本数: {len(X_test_scaled)}\n")
        f.write(f"特征维度: {CONFIG['input_dim']}\n")
        f.write(f"类别数量: {CONFIG['num_classes']}\n")
        f.write(f"数据标准化: 是 (基于训练集)\n\n")
        
        # 🔥 Gross Classification组织映射信息
        if 'gross_details' in val_results:
            tissue_mapping = val_results['gross_details']['tissue_mapping']
            f.write("Gross Classification组织映射:\n")
            f.write("-" * 30 + "\n")
            for tissue_group, class_indices in tissue_mapping.items():
                f.write(f"{tissue_group}: 类别 {class_indices[0]}-{class_indices[-1]} (共{len(class_indices)}个类别)\n")
            f.write("\n")
        
        # 性能评估结果
        f.write("性能评估结果:\n")
        f.write("-" * 30 + "\n")
        
        datasets = [("训练集", train_results), ("验证集", val_results), ("测试集", test_results)]
        for name, results in datasets:
            f.write(f"\n{name}:\n")
            f.write(f"  细分类准确率 (102类): {results['accuracy']:.6f}\n")
            f.write(f"  粗分类准确率 (6组): {results['gross_accuracy']:.6f}\n")  # 🔥 新增
            f.write(f"  准确率提升: +{results['gross_accuracy'] - results['accuracy']:.6f}\n")  # 🔥 新增
            f.write(f"  F1宏平均: {results['f1_macro']:.6f}\n")
            f.write(f"  F1加权平均: {results['f1_weighted']:.6f}\n")
            f.write(f"  平衡准确率: {results['balanced_accuracy']:.6f}\n")
            f.write(f"  Cohen's Kappa: {results['kappa']:.6f}\n")
            f.write(f"  损失: {results['loss']:.6f}\n")
            f.write(f"  预测类别数: {len(results['unique_classes'])}\n")
            
            # 🔥 新增：组织群性能分析
            if 'gross_details' in results:
                gross_details = results['gross_details']
                f.write(f"  组织群性能分析:\n")
                for tissue, acc in gross_details['gross_class_accuracies'].items():
                    f.write(f"    {tissue}: {acc:.6f}\n")
            
            # 类别性能统计
            f1_per_class = results['f1_per_class']
            f.write(f"  类别F1统计:\n")
            f.write(f"    平均值: {np.mean(f1_per_class):.6f}\n")
            f.write(f"    标准差: {np.std(f1_per_class):.6f}\n")
            f.write(f"    最小值: {np.min(f1_per_class):.6f}\n")
            f.write(f"    最大值: {np.max(f1_per_class):.6f}\n")
            
            # 最差和最好的类别
            worst_classes = np.argsort(f1_per_class)[:5]
            best_classes = np.argsort(f1_per_class)[-5:]
            
            f.write(f"    最差5个类别: {worst_classes.tolist()} (F1: {f1_per_class[worst_classes].tolist()})\n")
            f.write(f"    最好5个类别: {best_classes.tolist()} (F1: {f1_per_class[best_classes].tolist()})\n")
        
        # 🔥 数据集间比较 (包含gross accuracy)
        f.write(f"\n数据集间性能差异:\n")
        f.write("-" * 30 + "\n")
        f.write(f"细分类准确率差异:\n")
        f.write(f"  训练-验证: {train_results['accuracy'] - val_results['accuracy']:+.6f}\n")
        f.write(f"  验证-测试: {val_results['accuracy'] - test_results['accuracy']:+.6f}\n")
        f.write(f"粗分类准确率差异:\n")  # 🔥 新增
        f.write(f"  训练-验证: {train_results['gross_accuracy'] - val_results['gross_accuracy']:+.6f}\n")
        f.write(f"  验证-测试: {val_results['gross_accuracy'] - test_results['gross_accuracy']:+.6f}\n")
        f.write(f"F1宏平均差异:\n")
        f.write(f"  训练-验证: {train_results['f1_macro'] - val_results['f1_macro']:+.6f}\n")
        f.write(f"  验证-测试: {val_results['f1_macro'] - test_results['f1_macro']:+.6f}\n")
        
        # 相关性分析
        if len(train_results['f1_per_class']) == len(val_results['f1_per_class']):
            train_val_corr = np.corrcoef(train_results['f1_per_class'], val_results['f1_per_class'])[0, 1]
            f.write(f"训练-验证类别F1相关性: {train_val_corr:.6f}\n")
        
        if len(val_results['f1_per_class']) == len(test_results['f1_per_class']):
            val_test_corr = np.corrcoef(val_results['f1_per_class'], test_results['f1_per_class'])[0, 1]
            f.write(f"验证-测试类别F1相关性: {val_test_corr:.6f}\n")
        
        # 🔥 新增：Gross Classification深度分析
        f.write(f"\nGross Classification深度分析:\n")
        f.write("-" * 30 + "\n")
        
        # 整体提升分析
        datasets_names = ["训练集", "验证集", "测试集"]
        datasets_results = [train_results, val_results, test_results]
        
        f.write(f"细分类 vs 粗分类准确率对比:\n")
        for name, results in zip(datasets_names, datasets_results):
            fine_acc = results['accuracy']
            gross_acc = results['gross_accuracy']
            improvement = gross_acc - fine_acc
            improvement_pct = (improvement / fine_acc) * 100
            f.write(f"  {name}:\n")
            f.write(f"    细分类: {fine_acc:.6f}\n")
            f.write(f"    粗分类: {gross_acc:.6f}\n")
            f.write(f"    提升: +{improvement:.6f} ({improvement_pct:+.2f}%)\n")
        
        # 组织群难度分析 (基于验证集)
        if 'gross_details' in val_results:
            gross_details = val_results['gross_details']
            tissue_accs = gross_details['gross_class_accuracies']
            
            f.write(f"\n组织群难度排序 (基于验证集):\n")
            sorted_tissues = sorted(tissue_accs.items(), key=lambda x: x[1], reverse=True)
            for i, (tissue, acc) in enumerate(sorted_tissues):
                f.write(f"  {i+1}. {tissue}: {acc:.6f}\n")
            
            # 样本分布分析
            true_counts = gross_details['true_counts']
            pred_counts = gross_details['pred_counts']
            
            f.write(f"\n样本分布分析 (验证集):\n")
            f.write(f"组织群    真实样本数    预测样本数    分布差异\n")
            f.write("-" * 50 + "\n")
            for tissue in gross_details['gross_labels']:
                true_count = true_counts.get(tissue, 0)
                pred_count = pred_counts.get(tissue, 0)
                diff = pred_count - true_count
                f.write(f"{tissue:<12} {true_count:<10} {pred_count:<12} {diff:+d}\n")
        
        # 过拟合分析
        f.write(f"\n过拟合分析:\n")
        f.write("-" * 30 + "\n")
        
        # 细分类过拟合分析
        train_val_gap = train_results['accuracy'] - val_results['accuracy']
        if train_val_gap > 0.1:
            f.write("⚠️ 细分类可能存在过拟合 (训练准确率显著高于验证准确率)\n")
        elif train_val_gap > 0.05:
            f.write("⚠️ 细分类轻微过拟合倾向\n")
        else:
            f.write("✅ 细分类过拟合风险较低\n")
        
        # 🔥 粗分类过拟合分析
        train_val_gross_gap = train_results['gross_accuracy'] - val_results['gross_accuracy']
        if train_val_gross_gap > 0.1:
            f.write("⚠️ 粗分类可能存在过拟合\n")
        elif train_val_gross_gap > 0.05:
            f.write("⚠️ 粗分类轻微过拟合倾向\n")
        else:
            f.write("✅ 粗分类过拟合风险较低\n")
        
        val_test_gap = val_results['accuracy'] - test_results['accuracy']
        val_test_gross_gap = val_results['gross_accuracy'] - test_results['gross_accuracy']
        
        if abs(val_test_gap) < 0.02 and abs(val_test_gross_gap) < 0.02:
            f.write("✅ 验证集是测试集的良好代理 (细分类和粗分类都稳定)\n")
        elif val_test_gap > 0.05 or val_test_gross_gap > 0.05:
            f.write("⚠️ 验证集可能过于乐观\n")
        else:
            f.write("📊 验证集与测试集存在一定差异但可接受\n")
        
        # 🔥 关键发现总结
        f.write(f"\n关键发现总结:\n")
        f.write("-" * 30 + "\n")
        
        # 计算平均提升
        avg_improvement = np.mean([
            train_results['gross_accuracy'] - train_results['accuracy'],
            val_results['gross_accuracy'] - val_results['accuracy'],
            test_results['gross_accuracy'] - test_results['accuracy']
        ])
        
        f.write(f"1. 粗分类相比细分类平均提升 {avg_improvement:.4f} ({avg_improvement/np.mean([train_results['accuracy'], val_results['accuracy'], test_results['accuracy']])*100:.1f}%)\n")
        
        # 最佳和最差组织群
        if 'gross_details' in val_results:
            tissue_accs = val_results['gross_details']['gross_class_accuracies']
            best_tissue = max(tissue_accs, key=tissue_accs.get)
            worst_tissue = min(tissue_accs, key=tissue_accs.get)
            f.write(f"2. 最易分类组织群: {best_tissue} (准确率: {tissue_accs[best_tissue]:.4f})\n")
            f.write(f"3. 最难分类组织群: {worst_tissue} (准确率: {tissue_accs[worst_tissue]:.4f})\n")
        
        # 泛化性能
        fine_generalization = val_results['accuracy'] - test_results['accuracy']
        gross_generalization = val_results['gross_accuracy'] - test_results['gross_accuracy']
        
        if abs(gross_generalization) < abs(fine_generalization):
            f.write(f"4. 粗分类泛化性能更稳定 (验证-测试差异: 粗分类 {gross_generalization:+.4f} vs 细分类 {fine_generalization:+.4f})\n")
        else:
            f.write(f"4. 细分类泛化性能更稳定\n")
        
        # 🔥 组织群特异性分析
        if 'gross_details' in val_results and 'gross_details' in test_results:
            val_tissue_accs = val_results['gross_details']['gross_class_accuracies']
            test_tissue_accs = test_results['gross_details']['gross_class_accuracies']
            
            f.write(f"\n组织群特异性分析:\n")
            f.write("-" * 30 + "\n")
            f.write(f"组织群      验证准确率    测试准确率    泛化差异\n")
            f.write("-" * 50 + "\n")
            
            for tissue in val_tissue_accs.keys():
                val_acc = val_tissue_accs[tissue]
                test_acc = test_tissue_accs.get(tissue, 0)
                diff = val_acc - test_acc
                f.write(f"{tissue:<12} {val_acc:.6f}   {test_acc:.6f}   {diff:+.6f}\n")
            
            # 识别泛化最好和最差的组织群
            tissue_diffs = {tissue: val_tissue_accs[tissue] - test_tissue_accs.get(tissue, 0) 
                           for tissue in val_tissue_accs.keys()}
            
            best_generalization_tissue = min(tissue_diffs, key=lambda x: abs(tissue_diffs[x]))
            worst_generalization_tissue = max(tissue_diffs, key=lambda x: abs(tissue_diffs[x]))
            
            f.write(f"\n泛化性能分析:\n")
            f.write(f"  最稳定组织群: {best_generalization_tissue} (差异: {tissue_diffs[best_generalization_tissue]:+.4f})\n")
            f.write(f"  最不稳定组织群: {worst_generalization_tissue} (差异: {tissue_diffs[worst_generalization_tissue]:+.4f})\n")
        
        # 模型推荐
        f.write(f"\n模型应用建议:\n")
        f.write("-" * 30 + "\n")
        
        if avg_improvement > 0.1:
            f.write("🎯 强烈推荐: 对于需要稳定性的应用，优先考虑粗分类结果\n")
        elif avg_improvement > 0.05:
            f.write("📊 建议: 可以考虑提供粗分类作为辅助诊断\n")
        else:
            f.write("📋 信息: 粗分类提升有限，但仍可作为解释性工具\n")
        
        if 'gross_details' in val_results:
            worst_acc = min(val_results['gross_details']['gross_class_accuracies'].values())
            worst_tissue = min(val_results['gross_details']['gross_class_accuracies'], 
                             key=val_results['gross_details']['gross_class_accuracies'].get)
            if worst_acc < 0.7:
                f.write(f"⚠️ 注意: {worst_tissue} 组织群准确率较低 ({worst_acc:.3f})，需要额外关注\n")
        
        # 🔥 临床应用指导
        f.write(f"\n临床应用指导:\n")
        f.write("-" * 30 + "\n")
        
        if 'gross_details' in test_results:
            test_tissue_accs = test_results['gross_details']['gross_class_accuracies']
            high_confidence_tissues = [tissue for tissue, acc in test_tissue_accs.items() if acc > 0.9]
            moderate_confidence_tissues = [tissue for tissue, acc in test_tissue_accs.items() if 0.8 <= acc <= 0.9]
            low_confidence_tissues = [tissue for tissue, acc in test_tissue_accs.items() if acc < 0.8]
            
            f.write(f"按置信度分级:\n")
            if high_confidence_tissues:
                f.write(f"  高置信度 (>90%): {', '.join(high_confidence_tissues)}\n")
                f.write(f"    建议: 可直接用于临床辅助诊断\n")
            
            if moderate_confidence_tissues:
                f.write(f"  中等置信度 (80-90%): {', '.join(moderate_confidence_tissues)}\n")
                f.write(f"    建议: 结合其他信息进行综合判断\n")
            
            if low_confidence_tissues:
                f.write(f"  低置信度 (<80%): {', '.join(low_confidence_tissues)}\n")
                f.write(f"    建议: 需要额外验证，谨慎使用\n")
        
        # 🔥 数据质量评估
        f.write(f"\n数据质量评估:\n")
        f.write("-" * 30 + "\n")
        
        # 类别平衡性
        if 'gross_details' in val_results:
            true_counts = val_results['gross_details']['true_counts']
            total_samples = sum(true_counts.values())
            
            f.write(f"组织群样本分布 (验证集):\n")
            for tissue, count in true_counts.items():
                percentage = (count / total_samples) * 100
                f.write(f"  {tissue}: {count} 样本 ({percentage:.1f}%)\n")
            
            # 检查类别不平衡
            max_count = max(true_counts.values())
            min_count = min(true_counts.values())
            imbalance_ratio = max_count / min_count
            
            f.write(f"\n类别平衡性分析:\n")
            f.write(f"  最大/最小样本比: {imbalance_ratio:.2f}\n")
            if imbalance_ratio > 10:
                f.write("  ⚠️ 存在严重类别不平衡，建议使用类别权重或重采样\n")
            elif imbalance_ratio > 5:
                f.write("  ⚠️ 存在中等程度类别不平衡\n")
            else:
                f.write("  ✅ 类别分布相对平衡\n")
        
        # 🔥 模型可解释性建议
        f.write(f"\n模型可解释性建议:\n")
        f.write("-" * 30 + "\n")
        f.write("1. 使用粗分类结果作为第一层解释，便于医生理解\n")
        f.write("2. 对于高置信度的粗分类，可进一步展示细分类结果\n")
        f.write("3. 对于低置信度区域，建议显示不确定性并要求人工复核\n")
        f.write("4. 结合显著性分析，突出对分类决策最重要的脑区特征\n")
        
        if 'gross_details' in val_results:
            best_tissue = max(val_results['gross_details']['gross_class_accuracies'], 
                            key=val_results['gross_details']['gross_class_accuracies'].get)
            f.write(f"5. {best_tissue} 组织群表现最佳，可作为模型验证的基准\n")
    
    print(f"✅ 详细报告已保存: {report_path}")



In [ ]:
# ============================================================================
# 第七部分：显著性分析增强版（保持Alex的逻辑但增加更多分析）
# ============================================================================

def enhanced_saliency_analysis():
    """
    增强版显著性分析，添加更多统计和可视化
    """
    print("\n🔍 开始增强版显著性分析...")
    
    # 模拟utils.find_layer_idx
    def find_layer_idx(model, layer_name):
        if layer_name == 'visualized_layer':
            return len(list(model.children())) - 1
        return -1
    
    # 增强版显著性计算
    def visualize_saliency_enhanced(model, layer_index, filter_indices, seed_input, keepdims=True):
        """增强版显著性计算，添加更多分析"""
        model.eval()
        
        # 确保输入是正确的形状
        if seed_input.ndim == 1:
            seed_input = seed_input.reshape(1, -1)
        elif seed_input.ndim == 3:
            seed_input = seed_input.reshape(1, -1)
        
        seed_input_tensor = torch.FloatTensor(seed_input).to(device)
        seed_input_tensor.requires_grad_(True)
        
        output = model(seed_input_tensor)
        target_score = output[0, filter_indices]
        
        # 计算梯度
        model.zero_grad()
        target_score.backward(retain_graph=True)
        
        # 获取显著性
        saliency = seed_input_tensor.grad.data.abs()
        
        # 额外分析
        analysis = {
            'saliency': saliency.cpu().numpy(),
            'max_saliency': saliency.max().item(),
            'mean_saliency': saliency.mean().item(),
            'std_saliency': saliency.std().item(),
            'target_confidence': torch.softmax(output, dim=1)[0, filter_indices].item(),
            'prediction_confidence': torch.softmax(output, dim=1).max().item(),
            'predicted_class': torch.argmax(output, dim=1).item()
        }
        
        if keepdims:
            return analysis
        else:
            analysis['saliency'] = analysis['saliency'].squeeze()
            return analysis
    
    # 分析配置
    indices_to_visualize = [1000, 5000, 10000, 15000, 20000]  # 可以调整
    layer_index = find_layer_idx(model, 'visualized_layer')
    
    # 存储所有分析结果
    all_analyses = []
    
    print(f"🎯 开始分析 {len(indices_to_visualize)} 个样本...")
    
    for i, index_to_visualize in enumerate(indices_to_visualize):
        if index_to_visualize >= len(X_val_scaled):
            print(f"⚠️ 索引 {index_to_visualize} 超出验证集范围，跳过")
            continue
            
        # 获取输入
        input_spectrum = X_val_scaled[index_to_visualize, :]
        input_class = np.argmax(y_val[index_to_visualize, :])
        
        print(f"📊 分析样本 {i+1}/{len(indices_to_visualize)}: 真实类别 {input_class}")
        
        # 生成显著性分析
        analysis = visualize_saliency_enhanced(
            model, layer_index, 
            filter_indices=input_class, 
            seed_input=input_spectrum, 
            keepdims=True
        )
        
        # 添加额外信息
        analysis['true_class'] = input_class
        analysis['sample_index'] = index_to_visualize
        analysis['input_spectrum'] = input_spectrum
        all_analyses.append(analysis)
        
        # 创建增强版可视化
        plt.figure(figsize=(15, 10))
        
        # 主图：显著性和输入光谱
        plt.subplot(2, 2, (1, 2))
        saliency_flat = analysis['saliency'].flatten()
        
        plt.plot(saliency_flat, 'g-', linewidth=2, label=f'Saliency (max: {analysis["max_saliency"]:.4f})')
        plt.plot(scaler.inverse_transform(input_spectrum.reshape(1, -1)).flatten(), 
                'b-', alpha=0.4, label='Original Spectrum')
        plt.plot(input_spectrum, 'r-', alpha=0.6, linewidth=1, label='Normalized Spectrum')
        
        # 添加灰色区域（如果需要）
        plt.axvspan(0, 15, color='gray', alpha=0.2, label='Region 1')
        plt.axvspan(225, 230, color='gray', alpha=0.1, label='Region 2')
        
        plt.title(f'Sample {index_to_visualize}: True Class {input_class} | '
                 f'Predicted: {analysis["predicted_class"]} | '
                 f'Confidence: {analysis["target_confidence"]:.3f}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Feature Index')
        plt.ylabel('Value / Saliency')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 左下：显著性分布
        plt.subplot(2, 2, 3)
        plt.hist(saliency_flat, bins=50, alpha=0.7, color='green', edgecolor='black')
        plt.axvline(analysis['mean_saliency'], color='red', linestyle='--', 
                   label=f'Mean: {analysis["mean_saliency"]:.4f}')
        plt.axvline(analysis['mean_saliency'] + analysis['std_saliency'], 
                   color='orange', linestyle='--', 
                   label=f'Mean+Std: {analysis["mean_saliency"] + analysis["std_saliency"]:.4f}')
        plt.title('Saliency Distribution')
        plt.xlabel('Saliency Value')
        plt.ylabel('Frequency')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 右下：预测信息
        plt.subplot(2, 2, 4)
        
        # 获取模型对这个样本的完整预测
        with torch.no_grad():
            model.eval()
            input_tensor = torch.FloatTensor(input_spectrum.reshape(1, -1)).to(device)
            output = model(input_tensor)
            probabilities = torch.softmax(output, dim=1).cpu().numpy().flatten()
        
        # 显示前10个最高概率的类别
        top_10_indices = np.argsort(probabilities)[-10:][::-1]
        top_10_probs = probabilities[top_10_indices]
        
        colors = ['red' if idx == input_class else 'lightblue' for idx in top_10_indices]
        
        plt.bar(range(10), top_10_probs, color=colors, alpha=0.7)
        plt.xticks(range(10), top_10_indices, rotation=45)
        plt.title('Top 10 Predictions')
        plt.xlabel('Class Index')
        plt.ylabel('Probability')
        plt.grid(True, alpha=0.3)
        
        # 标注真实类别
        if input_class in top_10_indices:
            true_class_pos = list(top_10_indices).index(input_class)
            plt.text(true_class_pos, top_10_probs[true_class_pos] + 0.01, 
                    'TRUE', ha='center', va='bottom', fontweight='bold', color='red')
        
        plt.tight_layout()
        
        # 保存图像
        save_path = os.path.join(export_path, 'visualizations', 
                                f'enhanced_saliency_sample_{index_to_visualize}_class_{input_class}.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"  ✅ 保存显著性分析: {save_path}")
        print(f"  📊 预测类别: {analysis['predicted_class']}, 置信度: {analysis['target_confidence']:.4f}")
        print(f"  📊 显著性统计: Max={analysis['max_saliency']:.4f}, Mean={analysis['mean_saliency']:.4f}")
    
    # 生成显著性分析总结报告
    generate_saliency_summary_report(all_analyses)
    
    return all_analyses

def generate_saliency_summary_report(all_analyses):
    """生成显著性分析总结报告"""
    
    if not all_analyses:
        return
    
    report_path = os.path.join(export_path, 'saliency_analysis_summary.txt')
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("显著性分析总结报告\n")
        f.write("=" * 50 + "\n\n")
        
        f.write(f"分析样本数量: {len(all_analyses)}\n")
        f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        # 整体统计
        max_saliencies = [a['max_saliency'] for a in all_analyses]
        mean_saliencies = [a['mean_saliency'] for a in all_analyses]
        target_confidences = [a['target_confidence'] for a in all_analyses]
        prediction_confidences = [a['prediction_confidence'] for a in all_analyses]
        
        f.write("显著性统计:\n")
        f.write("-" * 30 + "\n")
        f.write(f"最大显著性 - 均值: {np.mean(max_saliencies):.6f}, 标准差: {np.std(max_saliencies):.6f}\n")
        f.write(f"平均显著性 - 均值: {np.mean(mean_saliencies):.6f}, 标准差: {np.std(mean_saliencies):.6f}\n")
        f.write(f"目标类别置信度 - 均值: {np.mean(target_confidences):.6f}, 标准差: {np.std(target_confidences):.6f}\n")
        f.write(f"最高预测置信度 - 均值: {np.mean(prediction_confidences):.6f}, 标准差: {np.std(prediction_confidences):.6f}\n\n")
        
        # 预测准确性
        correct_predictions = sum(1 for a in all_analyses if a['predicted_class'] == a['true_class'])
        f.write(f"预测准确性: {correct_predictions}/{len(all_analyses)} ({correct_predictions/len(all_analyses)*100:.1f}%)\n\n")
        
        # 单个样本详情
        f.write("单个样本详情:\n")
        f.write("-" * 30 + "\n")
        f.write(f"{'样本':<8} {'真实':<6} {'预测':<6} {'正确':<6} {'目标置信度':<12} {'最大显著性':<12}\n")
        f.write("-" * 70 + "\n")
        
        for a in all_analyses:
            correct = "✓" if a['predicted_class'] == a['true_class'] else "✗"
            f.write(f"{a['sample_index']:<8} {a['true_class']:<6} {a['predicted_class']:<6} "
                   f"{correct:<6} {a['target_confidence']:<12.4f} {a['max_saliency']:<12.4f}\n")
    
    print(f"✅ 显著性分析总结报告已保存: {report_path}")



In [ ]:
# ============================================================================
# 第八部分：主执行流程
# ============================================================================

def main():
    """主执行函数"""
    print("🚀 开始Alex版本增强训练流程...")
    
    # 1. 执行训练
    print("\n" + "="*60)
    print("第一阶段: 增强版训练")
    print("="*60)
    
    history = enhanced_training_loop()
    
    # 2. 绘制训练历史
    print("\n" + "="*60)
    print("第二阶段: 生成训练可视化")
    print("="*60)
    
    training_curves_path = os.path.join(export_path, 'visualizations', 'comprehensive_training_curves.png')
    plot_training_history_comprehensive(history, training_curves_path)
    print(f"✅ 训练曲线已保存: {training_curves_path}")
    
    # 3. 最终评估
    print("\n" + "="*60)
    print("第三阶段: 最终全面评估")
    print("="*60)
    
    train_results, val_results, test_results = final_comprehensive_evaluation()
    
    # 4. 显著性分析
    print("\n" + "="*60)
    print("第四阶段: 增强版显著性分析")
    print("="*60)
    
    saliency_analyses = enhanced_saliency_analysis()
    
    # 5. 保存最终配置和历史
    print("\n" + "="*60)
    print("第五阶段: 保存完整结果")
    print("="*60)
    
    # 保存训练历史
    history_path = os.path.join(export_path, 'training_history.json')
    with open(history_path, 'w') as f:
        # 转换numpy数组为列表以便JSON序列化
        json_history = {}
        for key, value in history.items():
            if isinstance(value, list):
                json_history[key] = value
            else:
                json_history[key] = [float(v) if not isinstance(v, (int, float)) else v for v in value]
        json.dump(json_history, f, indent=4)
    print(f"✅ 训练历史已保存: {history_path}")
    
    # 保存配置
    config_path = os.path.join(export_path, 'enhanced_config.json')
    with open(config_path, 'w') as f:
        json.dump(CONFIG, f, indent=4)
    print(f"✅ 配置已保存: {config_path}")
    
    # 最终总结
    print("\n" + "="*60)
    print("🎉 Alex版本增强训练完成!")
    print("="*60)
    print(f"📁 所有结果保存在: {export_path}")
    print(f"🏆 最佳验证F1: {max(history['val_f1_macro']):.4f}")
    print(f"🎯 最终测试F1: {test_results['f1_macro']:.4f}")
    print(f"🎯 最终测试准确率: {test_results['accuracy']:.4f}")
    print(f"📊 生成的可视化文件:")
    
    viz_dir = os.path.join(export_path, 'visualizations')
    if os.path.exists(viz_dir):
        for file in os.listdir(viz_dir):
            if file.endswith('.png'):
                print(f"  - {file}")
    
    print(f"📝 生成的报告文件:")
    for file in os.listdir(export_path):
        if file.endswith('.txt') or file.endswith('.json'):
            print(f"  - {file}")

# 运行主程序
if __name__ == "__main__":
    main()